In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import ripser
import persim

In [2]:
########## read the data ##########

processes = [f"run_{i:02d}_decayed_1" for i in range(1,12)]
kappa_3 = [0.8, 0.84, 0.88, 0.92, 0.96, 1, 1.04, 1.08, 1.12, 1.16, 1.2]
# Xsection_NoDecay = [3.7154566e-01, 3.617386e-01, 3.524371e-01, 3.4375437e-01, 3.3552887e-01, 3.273126e-01, 3.197475e-01, 3.126318e-01, 3.058909e-01, 2.993453e-01, 2.935034e-01]     ### hhvv unit: fb
Xsection_NoDecay = [5.5479319e-2, 5.433959e-2, 5.3013911e-2, 5.1688287e-2, 5.06150778e-2, 4.9453757e-2, 4.833858e-2, 4.740145e-2, 4.6342168e-2, 4.543942e-2, 4.4229705e-2]     ### hhmumu unit: fb
BR_hbb = 0.5824
Xsection = []
for i in range(11):
    Xsection.append(Xsection_NoDecay[i]*BR_hbb*BR_hbb)
    
Luminosity = 10000   ### unit: fb^-1
simulation_num = 100000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Mass", "VLCjetR10N2.Constituents", "EFlowTrack.fUniqueID", "EFlowTrack.PT", "EFlowTrack.Eta", "EFlowTrack.Phi", "EFlowTrack.PID", "EFlowPhoton.fUniqueID", "EFlowPhoton.ET", "EFlowPhoton.Eta", "EFlowPhoton.Phi", "EFlowNeutralHadron.fUniqueID", "EFlowNeutralHadron.ET", "EFlowNeutralHadron.Eta", "EFlowNeutralHadron.Phi"]

##### set data path #####

event_path = []
for type in processes:
    event_path.append("/data/mucollider/two_boosted/10TeV/scan_kappa3_hhmumu_small/Events/" + type + "/delphes_events.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print(process, "Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

/usr/local/lib/python3.8/dist-packages/awkward/array/base.py:394: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return cls.numpy.array(value, copy=False)


run_01_decayed_1 Time:10.4470 (s)
run_02_decayed_1 Time:19.4889 (s)
run_03_decayed_1 Time:29.3287 (s)
run_04_decayed_1 Time:38.4762 (s)
run_05_decayed_1 Time:47.8320 (s)
run_06_decayed_1 Time:57.4756 (s)
run_07_decayed_1 Time:65.5945 (s)
run_08_decayed_1 Time:75.4439 (s)
run_09_decayed_1 Time:85.8677 (s)
run_10_decayed_1 Time:94.0615 (s)
run_11_decayed_1 Time:102.2622(s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]


In [4]:
def get_particle_info(event):
    ##### set jet constituents ID #####

    leadingJ_constituents = event[features.index("VLCjetR10N2.Constituents")][0][:-1]
    subleadingJ_constituents= event[features.index("VLCjetR10N2.Constituents")][1][:-1]

    ##### construct infromation for all EFlow particles in this events and sort by PT #####

    all_ID = np.concatenate((event[features.index("EFlowTrack.fUniqueID")], event[features.index("EFlowPhoton.fUniqueID")], event[features.index("EFlowNeutralHadron.fUniqueID")]))
    all_PT = np.concatenate((event[features.index("EFlowTrack.PT")], event[features.index("EFlowPhoton.ET")], event[features.index("EFlowNeutralHadron.ET")]))
    all_Eta = np.concatenate((event[features.index("EFlowTrack.Eta")], event[features.index("EFlowPhoton.Eta")], event[features.index("EFlowNeutralHadron.Eta")]))
    all_Phi = np.concatenate((event[features.index("EFlowTrack.Phi")], event[features.index("EFlowPhoton.Phi")], event[features.index("EFlowNeutralHadron.Phi")]))
    all_PID = np.concatenate((event[features.index("EFlowTrack.PID")], np.full(len(event[features.index("EFlowPhoton.fUniqueID")]), 22), np.full(len(event[features.index("EFlowNeutralHadron.fUniqueID")]), 4)))

    #####                          categorize PID                      #####
    ##### 1:electron 2:muon 3:photon 4:neutral hadron 5:charged hadron #####

    condlist = [abs(all_PID)==11, abs(all_PID)==13, abs(all_PID)==22, abs(all_PID)==4]
    choicelist = [1, 2, 3, 4]
    all_PID = np.select(condlist, choicelist, 5)

    sort_order = np.argsort(all_PT)[::-1]   ### from large PT to low PT

    all_ID = all_ID[sort_order]
    all_PT = all_PT[sort_order]
    all_Eta = all_Eta[sort_order]
    all_Phi = all_Phi[sort_order]
    all_PID = all_PID[sort_order]

    ##### find Jet particle in EFlow particle #####
    if len(leadingJ_constituents)>20: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0][0:20]
    else: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0]

    if len(subleadingJ_constituents)>20: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0][0:20]
    else: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0]

    leadingJ_PT, subleadingJ_PT = all_PT[where_leading], all_PT[where_subleading]
    leadingJ_Eta, subleadingJ_Eta = all_Eta[where_leading], all_Eta[where_subleading]
    leadingJ_Phi, subleadingJ_Phi = all_Phi[where_leading], all_Phi[where_subleading]
    leadingJ_PID, subleadingJ_PID = all_PID[where_leading], all_PID[where_subleading]

    leadingJ_info, subleadingJ_info = np.stack((leadingJ_PT, leadingJ_Eta, leadingJ_Phi, leadingJ_PID), axis=1).ravel(), np.stack((subleadingJ_PT, subleadingJ_Eta, subleadingJ_Phi, subleadingJ_PID), axis=1).ravel()

    ##### do zero padding #####

    particle_info = np.zeros(2*20*4)
    particle_info[0:len(leadingJ_info)] = leadingJ_info
    particle_info[80:80+len(subleadingJ_info)] = subleadingJ_info
    
    return particle_info

In [5]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 100000 run_01_decayed_1 events. Corresponding cross section: 0.018818016896573444 (fb)
There are 100000 run_02_decayed_1 events. Corresponding cross section: 0.0184314324905984 (fb)
There are 100000 run_03_decayed_1 events. Corresponding cross section: 0.01798177574875136 (fb)
There are 100000 run_04_decayed_1 events. Corresponding cross section: 0.017532137662341123 (fb)
There are 100000 run_05_decayed_1 events. Corresponding cross section: 0.01716811609136333 (fb)
There are 100000 run_06_decayed_1 events. Corresponding cross section: 0.01677420796792832 (fb)
There are 100000 run_07_decayed_1 events. Corresponding cross section: 0.016395951348940802 (fb)
There are 100000 run_08_decayed_1 events. Corresponding cross section: 0.016078086449152002 (fb)
There are 100000 run_09_decayed_1 events. Corresponding cross section: 0.015718788841799683 (fb)
There are 100000 run_10_decayed_1 events. Corresponding cross section: 0.015412585964339203 (fb)
There are 100

In [6]:
##### generate particle information #####

particle_info = []

for process in processes:
    for event in events[processes.index(process)]:
        particle_info.append(get_particle_info(event))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:127.3561(s)
Time:132.4549(s)
Time:137.5498(s)
Time:142.6541(s)
Time:147.7011(s)
Time:152.7502(s)
Time:157.7404(s)
Time:162.7165(s)
Time:167.5293(s)
Time:172.3557(s)
Time:177.1784(s)


In [7]:
particle_info[0]

array([58.59396362, -1.50148296, -2.51384521,  5.        , 56.05474091,
       -1.50335228, -2.52058721,  5.        , 53.70986176, -1.59179032,
       -2.51075006,  4.        , 48.08950424, -1.5150013 , -2.51673388,
        5.        , 36.38004303, -1.50891078, -2.51834488,  3.        ,
       19.72415924, -1.49413836, -2.51466298,  5.        , 19.33868217,
       -1.20693135, -2.86738443,  5.        , 16.36065483, -1.44062901,
       -2.53364849,  5.        , 15.23096085, -1.18012679, -2.88405442,
        3.        , 15.10826397, -1.51298356, -2.5018692 ,  3.        ,
       14.49830627, -1.49991059, -2.54728484,  3.        , 14.21096611,
       -1.524593  , -2.52920938,  5.        , 12.95547199, -1.1313957 ,
       -2.87261081,  5.        , 11.97648335, -1.48364329, -2.54469466,
        5.        , 11.71167183, -1.17360342, -2.7380507 ,  5.        ,
       11.54250908, -1.1571238 , -2.87734771,  5.        ,  9.24921703,
       -1.21292162, -2.87116575,  3.        ,  9.18110847, -1.16

In [8]:
##### output test data #####

d = {'particle_info': particle_info}
df = pd.DataFrame(data=d)
df

,particle_info
0,"[58.593963623046875, -1.5014829635620117, -2.5..."
1,"[49.10063934326172, 2.1216278076171875, -2.908..."
2,"[84.83971405029297, 0.811669647693634, -0.2882..."
3,"[40.729705810546875, -0.5728055834770203, 1.98..."
4,"[65.59156036376953, 1.2869694232940674, 1.6059..."
...,...
267665,"[288.176513671875, 1.6106771230697632, -1.0309..."
267666,"[72.80271911621094, 1.3955888748168945, 1.1342..."
267667,"[71.59187316894531, 0.7271574139595032, -0.321..."
267668,"[66.90066528320312, 1.707480788230896, -0.4187..."


In [9]:
import h5py
h5f = h5py.File('/data/mucollider/two_boosted/10TeV/scan_kappa3_hhmumu_small_LL.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")
h5f.close()

已寫入 Dataset: particle_info
